# Analyze Channel Links via FORWARD_FROM

This notebook analyzes FORWARD_FROM relationships to examine links between channels - which channels forward messages from which other channels.


In [ ]:
# Install dependencies if needed
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Install required packages
%pip install -q neo4j pandas matplotlib seaborn networkx


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from neo4j import GraphDatabase
from src.disinfograph.config import get_neo4j_config

# Set up plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)


## Step 1: Connect to Neo4j


In [3]:
# Get Neo4j configuration
neo4j_config = get_neo4j_config()

# Connect to Neo4j
driver = GraphDatabase.driver(
    neo4j_config["uri"],
    auth=(neo4j_config["username"], neo4j_config["password"])
)

print(f"✓ Connected to Neo4j")
print(f"  URI: {neo4j_config['uri']}")
print(f"  Database: {neo4j_config['database']}")


✓ Connected to Neo4j
  URI: bolt://localhost:7687
  Database: neo4j


## Step 2: Analyze FORWARD_FROM Relationships Between Channels


In [4]:
# Query to find all FORWARD_FROM relationships and group by source/target channels
query = """
MATCH (src_msg:Message)-[:FORWARD_FROM]->(dst_msg:Message)
MATCH (src_ch:Channel)-[:POSTED]->(src_msg)
MATCH (dst_ch:Channel)-[:POSTED]->(dst_msg)
WHERE src_ch.channel_id <> dst_ch.channel_id  // Only inter-channel forwards
RETURN 
    src_ch.channel_id as source_channel_id,
    src_ch.username as source_channel_username,
    src_ch.title as source_channel_title,
    src_ch.label as source_channel_label,
    dst_ch.channel_id as target_channel_id,
    dst_ch.username as target_channel_username,
    dst_ch.title as target_channel_title,
    dst_ch.label as target_channel_label,
    count(*) as forward_count
ORDER BY forward_count DESC
"""

with driver.session() as session:
    result = session.run(query)
    forward_data = [dict(record) for record in result]

if forward_data:
    df = pd.DataFrame(forward_data)
    print(f"✓ Found {len(df):,} channel-to-channel forward relationships")
    print(f"\nTotal unique channel pairs: {len(df):,}")
    print(f"Total forwards: {df['forward_count'].sum():,}")
else:
    print("⚠️  No FORWARD_FROM relationships found between channels")
    df = pd.DataFrame()


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `FORWARD_FROM` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=27, offset=27>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 27, 'line': 2, 'column': 27}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (src_msg:Message)-[:FORWARD_FROM]->(dst_msg:Message)\nMATCH (src_ch:Channel)-[:POSTED]->(src_msg)\nMATCH (dst_ch:Channel)-[:POSTED]->(dst_msg)\nWHERE src_ch.channel_id <> dst_ch.channel_id  // Only inter-channel forwards\nRETURN \n    src_ch.channel_id as source_channel_id,\n    src_ch.u

⚠️  No FORWARD_FROM relationships found between channels


## Step 3: Display Channel Forward Network


In [5]:
if len(df) > 0:
    print("📊 Top Channel Forward Relationships:\n")
    print(df.head(20).to_string(index=False))
    
    print(f"\n\n📈 Summary Statistics:")
    print(f"  Channels that forward from others: {df['source_channel_id'].nunique():,}")
    print(f"  Channels that are forwarded from: {df['target_channel_id'].nunique():,}")
    print(f"  Average forwards per pair: {df['forward_count'].mean():.1f}")
    print(f"  Max forwards between a pair: {df['forward_count'].max():,}")
else:
    print("No data to display")


No data to display


## Step 4: Visualize Channel Forward Network


In [6]:
if len(df) > 0:
    # Create a bar chart of top forward relationships
    top_pairs = df.head(20).copy()
    top_pairs['pair'] = top_pairs.apply(
        lambda x: f"{x['source_channel_username'] or 'Unknown'} → {x['target_channel_username'] or 'Unknown'}", 
        axis=1
    )
    
    plt.figure(figsize=(14, 8))
    plt.barh(range(len(top_pairs)), top_pairs['forward_count'], color='steelblue', alpha=0.7)
    plt.yticks(range(len(top_pairs)), top_pairs['pair'])
    plt.xlabel('Number of Forwards', fontsize=12)
    plt.title('Top 20 Channel Forward Relationships', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    
    # Add value labels
    for i, v in enumerate(top_pairs['forward_count']):
        plt.text(v, i, f' {v:,}', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data to visualize")


No data to visualize


## Step 5: Channel Forward Statistics


In [7]:
if len(df) > 0:
    # Channels that forward the most
    forwarders = df.groupby(['source_channel_id', 'source_channel_username', 'source_channel_title', 'source_channel_label']).agg({
        'forward_count': 'sum'
    }).reset_index()
    forwarders = forwarders.sort_values('forward_count', ascending=False)
    
    print("📤 Channels That Forward the Most (from other channels):")
    print(forwarders.head(15).to_string(index=False))
    
    print("\n\n📥 Channels That Are Forwarded From the Most:")
    forwardees = df.groupby(['target_channel_id', 'target_channel_username', 'target_channel_title', 'target_channel_label']).agg({
        'forward_count': 'sum'
    }).reset_index()
    forwardees = forwardees.sort_values('forward_count', ascending=False)
    print(forwardees.head(15).to_string(index=False))
else:
    print("No data to analyze")


No data to analyze


## Step 6: Export Results


In [8]:
if len(df) > 0:
    # Export to CSV
    output_file = project_root / "data" / "channel_forward_relationships.csv"
    output_file.parent.mkdir(exist_ok=True)
    df.to_csv(output_file, index=False)
    print(f"✓ Exported results to: {output_file}")
    print(f"  Total rows: {len(df):,}")
else:
    print("No data to export")


No data to export


In [9]:
# Close Neo4j connection
driver.close()
print("✓ Connection closed")


✓ Connection closed
